# Stereo Vision
- 

# Depth Perception (깊이 인식)
- 

# Disparity (디스패리티)
- 왼쪽 사진의 이 픽셀이 오른쪽 사진에서는 어디에 있습니까? 왼쪽에서 보이는 위치와 오른쪽에서 보이는 위치가 다릅니다. 이 x좌표 차이를 계산하는 것이 바로 Disparity (디스패리티) 입니다. (Disparity = 왼쪽 픽셀 위치 - 오른쪽 픽셀 위치 / 200- 180 = 20)
- 왜 disparity가 중요하냐? => 거리(depth)를 계산할 수 있음
  * Depth = (focal_length × baseline) / disparity
  * focal_length = 카메라 렌즈 특성
  * baseline = 두 카메라 사이 거리
  * disparity = 두 이미지에서 물체 위치 차이
- 가까운 물체 : disparity = 큼, Depth = 작음
- 먼물체 : disparity = 작음, Depth = 큼

# Stereo Vision의 핵심 문제?
- 왼쪽 이미지의 픽셀이 오른쪽 이미지의 어느 픽셀인지 찾는 것 **Stereo Matching** 이라고도 함

# disp16.png
- 각 픽셀마다 disparity 값을 저장한 이미지

# Stereo Matching이 매우 어렵다.
1- 하늘 텍스처가 없습니다.
2- 반사, 유리, 물 픽셀이 달라집니다.
3- 왼쪽에서는 보이는데 오른쪽에서는 안 보입니다.

# 그래서 딥러닝, GA Net이 등장합니다.

# GA-Net
- 두 이미지를 입력으로 받아 각 픽셀의 disparity를 예측하는 딥러닝 모델
- 입력 : Left Image, Right Image
- 모델 : GA-Net
- 출력 : Disparity Map

# 전체 파이프라인
Left image
Right image
      ↓
Stereo Matching (GA-Net)
      ↓
Disparity Map
      ↓
Depth 계산
      ↓
3D geometry
      ↓
Slope 계산

# 매우 중요한 이해 포인트
- 여러분 프로젝트는 이미 disparity가 계산된 상태입니다.


1. 두 카메라는 서로 떨어져 있습니다.
```
Camera L        Camera R
    |--------------|
         baseline
```

여기서 baseline은 120 mm 정도 입니다(conf 파일에서 확인 가능, BaseLine = 120.009)

2. 삼각형이 만들어집니다
```
CameraL ----- CameraR
     \        /
      \      /
       \    /
        물체
```
이 삼각형에서

```
disparity
baseline
focal length
```
를 알면 물체 거리(depth)를 계산할 수 있습니다.

```
Depth = (focal_length × baseline) / disparity
```
3. conf 파일이 왜 필요한가
Depth 계산에는
```
focal length
baseline
```
이 필요합니다.

이 값들이 여기 있습니다.
```
[LEFT_CAM_2K]
fx=1394.83
fy=1394.83
```

```
focal length	카메라 렌즈 특성
baseline	두 카메라 사이 거리
disparity	좌우 픽셀 차이
```

```
fx = focal length
```

```
BaseLine = 120.009
```

4. 이제 3D 좌표를 만들 수 있습니다.
이미지 픽셀은
```
(x, y)
```
입니다. 

Depth가 추가되면
```
(x, y, z)
```
가 됩니다.

즉 **3D 좌표** 입니다.
이걸 **Point Cloud**라고 합니다.

5. Point Cloud가 왜 중요한가?
- 3D 공간에서 도로는 평면입니다.
- 그래서 우리는 도로 평면을 찾습니다.

6. Plane을 찾는 방법
- 이 점들을 보면 **기울어진 평면**이 위에 있습니다.
- 그래서 **Plane fitting**이라는 알고리즘을 사용합니다. 대표적으로 RANSAC을 사용합니다.

7. ROI는 무엇인가
ROI = Region Of Interest
> 우리가 관심 있는 영역
예를 들어 이미지가 이렇게 있다고 합시다.
```
┌────────────────────┐
│        하늘        │
│                    │
│        건물        │
│                    │
│        사람        │
│                    │
│        도로        │
└────────────────────┘
```
하지만 우리는 도로의 경사만 알고 싶습니다. 그래서 실제로 필요한 영역은
```
┌────────────────────┐
│                    │
│                    │
│                    │
│                    │
│        도로        │
└────────────────────┘
```

그래서 코드에서 이렇게 합니다.
```
roi = (0.6, 1.0)
```
✔ 계산량도 줄어듦


1. Segmentation 기반 장애물 마스킹이 무슨 의미인가?
- 사진에서 “도로/인도(지면)”만 남기고, 사람/차/벽/건물 같은 “지면이 아닌 것”을 가려(=마스킹) 버리는 것입니다.
2. Semantic Segmentation?
- 사진의 모든 픽셀에 대해 “이 픽셀이 무엇인지” 라벨을 붙이는 작업

PIDNet이 하는 일
- 픽셀별 클래스 맵(예: road/sidewalk/person…)
- PIDNet의 “유명한 특징”은: real-time을 목표로 설계돼서, 정확도만 높은 모델보다 빠르게 돌아가도록 최적화되어 있다는 점입니다. https://openaccess.thecvf.com/content/CVPR2023/papers/Xu_PIDNet_A_Real-Time_Semantic_Segmentation_Network_Inspired_by_PID_Controllers_CVPR_2023_paper.pdf?utm_source=chatgpt.com

In [ ]:
# PIDNet 공식 구현/가중치는 GitHub에 있고, 실사용은 “사전학습 가중치로 추론만” 하는 방식이 가장 현실적입니다.
# Qualcomm이 배포한 **PidNet(HuggingFace)**에는 ONNX 모델도 있어 웹데모/CPU 추론에 유리합니다
# Cityscapes 계열 segmentation은 흔히 **19-class trainId(0~18)**를 쓰며, 일반적으로 road, sidewalk 같은 클래스가 포함됩니다. (단, “id” vs “trainId”가 다를 수 있어 모델 출력 라벨 정의를 확인해야 함)
import os
import cv2
import numpy as np
import onnxruntime as ort

MODEL_PATH = "models/pidnet.onnx" # ONNX 모델을 로드할 때 사용됩니다.

# Cityscapes 계열 19-class trainId를 쓰는 경우가 흔함:
# (많은 구현에서) road=0, sidewalk=1 로 학습되는 경우가 많습니다.
# 하지만 모델에 따라 다를 수 있으니, 결과가 이상하면 여기부터 의심하세요.
ROAD_ID = 0
SIDEWALK_ID = 1

'''
0 = road
1 = sidewalk
2 = building
3 = person
4 = car
...
'''

# 이미지 정규화(normalization)
# 전처리: ImageNet mean/std (대부분 segmentation backbone에서 사용)
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Segmentation 모델은 특정 입력 해상도에서 학습됨.
# PIDNet/Cityscapes는 보통 (H,W)=(1024,2048)로 학습된 경우가 많아
# 디버깅/재현성 위해 일단 resize 권장.
TARGET_H, TARGET_W = 1024, 2048


# 모델이 먹을 수 있는 형태로 변환 (OpenCV 이미지 → 모델 입력 텐서)
# bgr = OpenCV 이미지
# shape = (H,W,3)
def preprocess_bgr(bgr: np.ndarray) -> tuple[np.ndarray, tuple[int, int]]:
    """BGR uint8 -> (1,3,H,W) float32"""
    orig_h, orig_w = bgr.shape[:2]
    img = cv2.resize(bgr, (TARGET_W, TARGET_H), interpolation=cv2.INTER_LINEAR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - MEAN) / STD
    img = np.transpose(img, (2, 0, 1))  # CHW
    img = img[None, ...]                # NCHW
    return img.astype(np.float32), (orig_h, orig_w)


def postprocess_logits(logits: np.ndarray, orig_hw: tuple[int, int]) -> np.ndarray:
    """
    logits: (1, C, H, W) or (1, H, W) depending on export.
    return: pred_label (orig_h, orig_w) int32
    """
    if logits.ndim == 4:
        pred = np.argmax(logits[0], axis=0).astype(np.int32)  # (H,W)
    elif logits.ndim == 3:
        # (1,H,W) already label?
        pred = logits[0].astype(np.int32)
    else:
        raise ValueError(f"Unexpected logits shape: {logits.shape}")

    # 원본 크기로 되돌림
    orig_h, orig_w = orig_hw
    pred = cv2.resize(pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    return pred


def make_ground_mask(pred_label: np.ndarray) -> np.ndarray:
    """road/sidewalk만 True인 바이너리 마스크"""
    mask = (pred_label == ROAD_ID) | (pred_label == SIDEWALK_ID)
    return mask.astype(np.uint8)  # 0/1


def overlay_mask(bgr: np.ndarray, mask01: np.ndarray) -> np.ndarray:
    """시각화: 지면(1) 영역만 초록빛으로 덮기"""
    overlay = bgr.copy()
    green = np.zeros_like(bgr, dtype=np.uint8)
    green[:, :, 1] = 255
    alpha = 0.35
    idx = mask01.astype(bool)
    overlay[idx] = (overlay[idx] * (1 - alpha) + green[idx] * alpha).astype(np.uint8)
    return overlay


def main(image_path: str, out_dir: str = "outputs"):
    os.makedirs(out_dir, exist_ok=True)

    bgr = cv2.imread(image_path)
    if bgr is None:
        raise FileNotFoundError(image_path)

    # ONNX 세션
    sess = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    # 전처리
    x, orig_hw = preprocess_bgr(bgr)

    # 추론
    y = sess.run([output_name], {input_name: x})[0]

    # 후처리
    pred_label = postprocess_logits(y, orig_hw)
    ground_mask01 = make_ground_mask(pred_label)

    # 저장물
    # 1) 라벨 맵(정수)을 png로 저장하려면 8-bit로 캐스팅(클래스 수 19라 가능)
    label_vis = pred_label.astype(np.uint8)
    mask_vis = (ground_mask01 * 255).astype(np.uint8)

    cv2.imwrite(os.path.join(out_dir, "pred_label.png"), label_vis)
    cv2.imwrite(os.path.join(out_dir, "ground_mask.png"), mask_vis)
    cv2.imwrite(os.path.join(out_dir, "overlay.png"), overlay_mask(bgr, ground_mask01))

    print("Saved:")
    print(" - pred_label.png (각 픽셀 클래스 ID)")
    print(" - ground_mask.png (road/sidewalk=255)")
    print(" - overlay.png (원본 위에 마스크 덮어쓴 시각화)")


if __name__ == "__main__":
    # 예: python infer_pidnet.py datas/Depth_001/ZED1_KSC_001032_left.png
    import sys
    if len(sys.argv) < 2:
        print("Usage: python infer_pidnet.py <image_path>")
        raise SystemExit(1)
    main(sys.argv[1])

FileNotFoundError: --f=c:\Users\user\AppData\Roaming\jupyter\runtime\kernel-v3641cf00c6963879964c0491de12d2b43c0400551.json

In [6]:
import os
import re
import math
import cv2
import numpy as np

# ---------------------------
# 1) .conf 파서 (필요한 값만)
# ---------------------------
def parse_zed_conf(conf_path, section="LEFT_CAM_FHD"):
    """
    section은 이미지 해상도에 맞춰 골라야 합니다.
    너희 crop가 1920x592면 보통 FHD 기준을 많이 씁니다.
    """
    data = {}
    current = None
    with open(conf_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"\[(.+)\]", line)
            if m:
                current = m.group(1)
                data[current] = {}
                continue
            if "=" in line and current is not None:
                k, v = line.split("=", 1)
                data[current][k.strip()] = float(v.strip())

    # intrinsics
    fx = data[section]["fx"]
    fy = data[section]["fy"]
    cx = data[section]["cx"]
    cy = data[section]["cy"]

    # baseline (보통 mm로 제공되는 경우가 많음)
    baseline_mm = data["STEREO"]["BaseLine"]
    baseline_m = baseline_mm / 1000.0

    return fx, fy, cx, cy, baseline_m

# ---------------------------
# 2) disp16 -> disparity(float) -> depth(m)
# ---------------------------
def disp16_to_depth(disp16_u16, fx, baseline_m, disp_scale="auto", min_disp=1.0):
    """
    disp16_u16: cv2.imread(..., IMREAD_UNCHANGED)로 읽은 uint16 disparity 이미지
    disp_scale:
      - "auto": 값 분포 보고 (x16 고정소수점)인지 추정
      - 1.0: 그대로 disparity 픽셀
      - 16.0: disp16이 disparity*16 형태일 때
    """
    disp = disp16_u16.astype(np.float32)

    if disp_scale == "auto":
        # 휴리스틱: 값이 수백~수만이면 *16일 가능성이 큼
        # disparity(픽셀)는 보통 0~256(상황 따라 다름) 근처가 흔함
        p95 = np.percentile(disp[disp > 0], 95) if np.any(disp > 0) else 0
        # p95가 512보다 크면 *16일 가능성이 매우 높다고 가정
        scale = 16.0 if p95 > 512 else 1.0
    else:
        scale = float(disp_scale)

    disparity = disp / scale

    # depth = fx * baseline / disparity
    # disparity가 0 또는 너무 작으면 무한대/폭발하니 마스킹
    depth = np.zeros_like(disparity, dtype=np.float32)
    valid = disparity >= float(min_disp)
    depth[valid] = (fx * baseline_m) / disparity[valid]  # meters

    return disparity, depth, scale

# ---------------------------
# 3) depth -> point cloud (ROI만)
# ---------------------------
def depth_to_points(depth_m, fx, fy, cx, cy, mask=None, roi=(0.6, 1.0), return_uv=False):
    H, W = depth_m.shape
    y0 = int(H * roi[0])
    y1 = int(H * roi[1])

    ys, xs = np.mgrid[y0:y1, 0:W]
    z = depth_m[y0:y1, :]

    if mask is not None:
        m = mask[y0:y1, :]
        valid = (z > 0) & m
    else:
        valid = (z > 0)

    xs_v = xs[valid].astype(np.float32)
    ys_v = ys[valid].astype(np.float32)
    z_v  = z[valid].astype(np.float32)

    X = (xs_v - cx) * z_v / fx
    Y = (ys_v - cy) * z_v / fy
    pts = np.stack([X, Y, z_v], axis=1)

    if return_uv:
        # u=x, v=y (이미지 픽셀 좌표)
        uv = np.stack([xs_v.astype(np.int32), ys_v.astype(np.int32)], axis=1)
        return pts, uv

    return pts

# ---------------------------
# 4) Plane fitting with RANSAC
# plane: ax + by + cz + d = 0
# ---------------------------
def fit_plane_ransac(points, n_iter=200, dist_thresh=0.03, sample_size=3, seed=42):
    """
    dist_thresh: meters (3cm 등). depth 품질에 따라 조절.
    반환: (a,b,c,d), inlier_mask
    """
    rng = np.random.default_rng(seed)
    N = points.shape[0]
    if N < 50:
        raise ValueError(f"Too few points for plane fitting: {N}")

    best_inliers = None
    best_count = -1
    best_plane = None

    for _ in range(n_iter):
        idx = rng.choice(N, size=sample_size, replace=False)
        p1, p2, p3 = points[idx]

        # 두 벡터의 외적으로 평면 법선
        v1 = p2 - p1
        v2 = p3 - p1
        n = np.cross(v1, v2)
        norm = np.linalg.norm(n)
        if norm < 1e-6:
            continue
        n = n / norm
        a, b, c = n
        d = -np.dot(n, p1)

        # 점-평면 거리
        dist = np.abs(points @ n + d)
        inliers = dist < dist_thresh
        count = int(np.sum(inliers))

        if count > best_count:
            best_count = count
            best_inliers = inliers
            best_plane = (a, b, c, d)

    if best_plane is None:
        raise ValueError("Failed to fit plane.")

    # (선택) inlier들로 least squares 재추정해서 더 안정화
    inlier_pts = points[best_inliers]
    # plane normal via SVD: minimize ||(p - mean)·n||
    centroid = inlier_pts.mean(axis=0)
    U, S, Vt = np.linalg.svd(inlier_pts - centroid, full_matrices=False)
    n = Vt[-1]
    n = n / (np.linalg.norm(n) + 1e-9)
    d = -np.dot(n, centroid)
    a, b, c = n

    return (a, b, c, d), best_inliers

def overlay_plane_inliers_on_image(img_bgr, uv, inlier_mask, alpha=0.45, color=(0, 255, 0)):
    """
    img_bgr: 원본 이미지(BGR)
    uv: (N,2) int32, [u,v]
    inlier_mask: (N,) bool
    """
    overlay = img_bgr.copy()
    uvs = uv[inlier_mask]

    # inlier 픽셀을 찍어주기 (점이 촘촘하면 면처럼 보임)
    overlay[uvs[:, 1], uvs[:, 0]] = color

    # 살짝 블러를 주면 점들이 면처럼 더 보기 좋아짐(선택)
    overlay = cv2.GaussianBlur(overlay, (0, 0), 1.2)

    out = cv2.addWeighted(overlay, alpha, img_bgr, 1 - alpha, 0)
    return out

# 0) 로드
conf_path = "datas/Depth_007/Depth_007.conf"
fx, fy, cx, cy, baseline_m = parse_zed_conf(conf_path, section="LEFT_CAM_FHD")

img = cv2.imread("datas/Depth_007/ZED4_KSC_010821_left.png")  # or crop_left
disp16 = cv2.imread("datas/Depth_007/ZED4_KSC_010821_disp16.png", cv2.IMREAD_UNCHANGED)

conf = cv2.imread("datas/Depth_007/ZED4_KSC_010821_confidence.png", cv2.IMREAD_GRAYSCALE)
mask = (conf > 0)  # confidence가 0/255라면 이게 맞음

# 1) disp16 -> depth
disparity, depth, used_scale = disp16_to_depth(disp16, fx, baseline_m, disp_scale="auto")

# 2) depth -> points (+uv)
pts, uv = depth_to_points(depth, fx, fy, cx, cy, mask=mask, roi=(0.6, 1.0), return_uv=True)

# 3) plane fit
plane, inliers = fit_plane_ransac(pts, n_iter=200, dist_thresh=0.03)

# 4) overlay
vis = overlay_plane_inliers_on_image(img, uv, inliers, alpha=0.5, color=(0, 255, 0))

cv2.imwrite("plane_inliers_overlay.png", vis)
print("saved: plane_inliers_overlay.png", "scale_used:", used_scale, "inlier_ratio:", inliers.mean())

saved: plane_inliers_overlay.png scale_used: 16.0 inlier_ratio: 0.7795331758542461


In [ ]:
# ---------------------------
# 5) Plane normal -> slope (%)
# ---------------------------
def plane_to_slope_percent(plane, assume_camera_y_down=True):
    """
    카메라 좌표계에서 '수직 방향'을 정의해야 slope가 나옵니다.
    보통 이미지 좌표 v가 아래로 증가하므로 camera Y가 아래 방향일 때가 많습니다.
    여기서는 '수직축'을 +Y 또는 -Y 중 하나로 가정.
    """
    a, b, c, d = plane
    n = np.array([a, b, c], dtype=np.float32)
    n = n / (np.linalg.norm(n) + 1e-9)

    # '수직(중력) 방향' 벡터 g를 가정
    # camera y-down이면 중력은 +Y 방향이라고 가정
    g = np.array([0.0, 1.0, 0.0], dtype=np.float32) if assume_camera_y_down else np.array([0.0, -1.0, 0.0], dtype=np.float32)

    # 평면과 수평의 관계를 직관적으로:
    # 평평한 지면이라면 normal은 중력 방향 g와 평행(또는 반평행)
    # 경사가 있으면 normal이 g에서 기울어짐
    cosang = np.clip(np.abs(np.dot(n, g)), 0.0, 1.0)
    theta = math.acos(cosang)  # 0이면 평평, 커질수록 기울어짐

    slope_percent = math.tan(theta) * 100.0
    return slope_percent, theta

def slope_to_level(slope_percent):
    # 너희 프로젝트 기준(초기안): ADA 8.33%를 중요한 경계로 사용
    if slope_percent < 2:
        return 0
    elif slope_percent < 5:
        return 1
    elif slope_percent < 8.33:
        return 2
    elif slope_percent < 12:
        return 3
    else:
        return 4

# ---------------------------
# 6) 한 장면 실행 함수
# ---------------------------
def run_one_scene(folder, prefix, conf_path, conf_section="LEFT_CAM_FHD"):
    """
    folder: Depth_007 폴더 경로
    prefix: 예) "ZED4_KSC_010821"
    """
    # 파일 경로 후보들
    disp16_path = os.path.join(folder, f"{prefix}_disp16.png")
    confmask_path = os.path.join(folder, f"{prefix}_confidence.png")  # binary
    left_path_candidates = [
        os.path.join(folder, f"{prefix}_left.png"),
        os.path.join(folder, f"{prefix}_L.png"),
    ]

    left_path = next((p for p in left_path_candidates if os.path.exists(p)), None)
    if left_path is None:
        raise FileNotFoundError("Left image not found (_left.png or _L.png).")

    if not os.path.exists(disp16_path):
        raise FileNotFoundError(f"disp16 not found: {disp16_path}")
    if not os.path.exists(confmask_path):
        raise FileNotFoundError(f"confidence mask not found: {confmask_path}")

    # 이미지 로드
    left = cv2.imread(left_path, cv2.IMREAD_COLOR)
    disp16 = cv2.imread(disp16_path, cv2.IMREAD_UNCHANGED)  # uint16 expected
    confmask = cv2.imread(confmask_path, cv2.IMREAD_GRAYSCALE)

    if disp16 is None or disp16.dtype != np.uint16:
        raise ValueError("disp16 image must be uint16. Check file reading.")
    if confmask is None:
        raise ValueError("confidence mask load failed.")

    # binary mask 만들기: 흰색(>0)을 True로
    mask = confmask > 0

    # 카메라 파라미터
    fx, fy, cx, cy, baseline_m = parse_zed_conf(conf_path, section=conf_section)

    # disparity -> depth
    disparity, depth_m, used_scale = disp16_to_depth(disp16, fx, baseline_m, disp_scale="auto", min_disp=1.0)

    # points 생성 (하단 ROI + confidence 마스크)
    pts = depth_to_points(depth_m, fx, fy, cx, cy, mask=mask, roi=(0.60, 1.00))

    # plane fitting
    plane, inliers = fit_plane_ransac(pts, n_iter=250, dist_thresh=0.03)

    # slope 계산
    slope_percent, theta = plane_to_slope_percent(plane, assume_camera_y_down=True)
    level = slope_to_level(slope_percent)

    result = {
        "prefix": prefix,
        "disp_scale_used": used_scale,
        "num_points": int(pts.shape[0]),
        "num_inliers": int(np.sum(inliers)),
        "inlier_ratio": float(np.sum(inliers) / max(1, pts.shape[0])),
        "slope_percent": float(slope_percent),
        "theta_degrees": float(theta * 180.0 / math.pi),
        "level": int(level),
        "plane": tuple(float(x) for x in plane),
    }
    return result, left, depth_m, mask

# ---------------------------
# 7) 실행 예시
# ---------------------------
if __name__ == "__main__":
    folder = "datas/Depth_007"
    conf_path = "datas/Depth_007/Depth_007.conf"
    prefix = "ZED4_KSC_010821"

    result, left_img, depth_m, mask = run_one_scene(
        folder=folder,
        prefix=prefix,
        conf_path=conf_path,
        conf_section="LEFT_CAM_FHD"  # crop가 FHD 기반이면 이거부터 시도
    )
    print(result)